# Experimento: Auditoría con Merkle Trees
El objetivo es simular un sistema de auditoría que garantice la integridad de eventos (como en una página web) usando Merkle Trees.

## Analogía con una página web
Una web real registra eventos como login de usuario, documentos firmados, cambios en datos, etc. Estos eventos se almacenan y deben ser inmutables (nadie los debe alterar).

El problema es que si alguien modifica un registro, ¿cómo lo detectamos? Nuestra propuesta es usar Merkle Trees para garantizar integridad.

In [ ]:
'''
La idea es que en lugar de guardar datos directamente:

- guardamos hashes
- los combinamos jerárquicamente
- obtenemos un Merkle Root

Si cualquier dato cambia, cambia todo el árbol.
'''
import hashlib
from typing import List

# Función de hash (SHA-256)
def hash_data(data: str) -> str:
    """
    Convierte un string en un hash SHA-256.
    
    Args:
        data (str): información a hashear
    
    Returns:
        str: hash hexadecimal
    """
    return hashlib.sha256(data.encode()).hexdigest()


## Construcción del Merkle Tree

In [ ]:
def build_merkle_tree(events: List[str]) -> List[List[str]]:
    """
    Construye un Merkle Tree a partir de eventos.
    
    Args:
        events (List[str]): lista de eventos
    
    Returns:
        List[List[str]]: niveles del árbol
    """
    
    # Nivel 0: hojas (hash de eventos)
    current_level = [hash_data(event) for event in events]
    tree = [current_level]

    # Construcción hacia arriba
    while len(current_level) > 1:
        next_level = []

        for i in range(0, len(current_level), 2):
            left = current_level[i]

            # Si hay número impar, duplicamos el último nodo
            right = current_level[i + 1] if i + 1 < len(current_level) else left

            # Concatenamos hashes hijos
            combined = left + right

            # Hash del nodo padre
            parent_hash = hash_data(combined)
            next_level.append(parent_hash)

        tree.append(next_level)
        current_level = next_level

    return tree


# Obtener la raíz del árbol
def get_merkle_root(tree: List[List[str]]) -> str:
    """
    Obtiene el hash raíz del Merkle Tree.
    """
    return tree[-1][0]

In [ ]:
##  Simulación de eventos de una web


In [ ]:
# Simulación de eventos de una web

events = [
    "User A logged in",
    "User B signed document X",
    "User C updated profile",
    "Admin approved request"
]

# Construimos árbol original
tree = build_merkle_tree(events)
original_root = get_merkle_root(tree)

print("🌳 Merkle Root original:")
print(original_root)


# ============================================
# ⚠️ Simulación de ataque/modificación
# ============================================

events[1] = "User B signed document X (MODIFIED)"

new_tree = build_merkle_tree(events)
new_root = get_merkle_root(new_tree)

print("\n🌳 Nuevo Merkle Root:")
print(new_root)


# ============================================
# 🔍 Verificación de integridad
# ============================================

if original_root != new_root:
    print("\n⚠️ ALERTA: Se detectó una modificación en los registros")
else:
    print("\n✅ Datos íntegros")